# negforge: 1D ballistic MOSFET simulator

This notebook is the Python frontend for the `negforge` Rust engine (`crates/negforge-core`, exposed here via PyO3 as the `negforge` package). It reproduces every plot from the original MATLAB coursework code in `legacy_matlab/` (`plot_potential`, `plot_Vg_I`, `plot_Vds_I`, the NEGF local-density-of-states heatmap) plus a self-consistent Poisson&harr;NEGF solve that the original code never had.

**Before running:** build and install the extension once from the repository root:
```bash
pip install maturin
maturin develop --release
```
See the top-level `README.md` for the physics background, unit conventions, and a list of deliberate deviations from the original MATLAB model (most importantly: the charge-feedback loop and two bug fixes in the NEGF charge-density formula).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import negforge

plt.rcParams["figure.figsize"] = (7, 4.5)

## 1. Electrostatic potential profile

A device with a gate and drain bias applied. `calc_potential()` is the **decoupled** solve (matches the original `calc_potential()`: charge density `rho` fixed at zero).

In [ ]:
dev = negforge.Device(v_ds=0.3, v_g=0.2)
dev.calc_potential()

plt.plot(dev.positions_nm, dev.psi_f)
plt.xlabel("position (nm)")
plt.ylabel(r"$\Psi_f$ (eV)")
plt.title("Decoupled electrostatic potential (V$_g$=0.2V, V$_{ds}$=0.3V)")
plt.grid(alpha=0.3)
plt.show()

print(dev)

## 2. Self-consistent Poisson&harr;NEGF solve

This closes the feedback loop the original MATLAB code never implemented: the NEGF-computed electron charge is fed back into the electrostatic solve and iterated to convergence. Compare against the decoupled potential above &mdash; the self-consistent charge screens (flattens) the barrier somewhat.

In [ ]:
dev_decoupled = negforge.Device(v_ds=0.3, v_g=0.2).calc_potential()
psi_decoupled = dev_decoupled.psi_f.copy()

dev_sc = negforge.Device(v_ds=0.3, v_g=0.2)
iterations, residual = dev_sc.solve_self_consistent()
print(f"converged in {iterations} iterations, residual={residual:.3e} eV")

plt.plot(dev_decoupled.positions_nm, psi_decoupled, label="decoupled (rho=0)")
plt.plot(dev_sc.positions_nm, dev_sc.psi_f, label="self-consistent")
plt.xlabel("position (nm)")
plt.ylabel(r"$\Psi_f$ (eV)")
plt.title("Decoupled vs. self-consistent potential")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 3. Transfer characteristic (I&ndash;V$_g$) and subthreshold swing

Mirrors `plot_Vg_I` / the `polyfit`-based swing extraction in the original code. The self-consistent sweep is slower (a full Poisson&ndash;NEGF solve per point) so it uses fewer points than the decoupled one.

In [ ]:
dev = negforge.Device(v_ds=0.3)
curve_decoupled = dev.sweep_v_g(0.0, 0.6, 0.01, self_consistent=False)

dev_sc = negforge.Device(v_ds=0.3)
curve_sc = dev_sc.sweep_v_g(0.0, 0.4, 0.05, self_consistent=True)

fig, ax = plt.subplots()
ax.semilogy(curve_decoupled.voltage, curve_decoupled.current, label="decoupled")
ax.semilogy(curve_sc.voltage, curve_sc.current, "o-", label="self-consistent")
ax.set_xlabel(r"$V_g$ (V)")
ax.set_ylabel("current (a.u., see README for units)")
ax.set_title("Transfer characteristic")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.show()

s_decoupled = curve_decoupled.subthreshold_swing(0.0, 0.4)
s_sc = curve_sc.subthreshold_swing(0.0, 0.4)
print(f"subthreshold swing (decoupled):        {s_decoupled * 1000:.1f} mV/decade")
print(f"subthreshold swing (self-consistent):  {s_sc * 1000:.1f} mV/decade")
print("(ideal thermal limit at 300 K is ln(10)*kT/e = 59.6 mV/decade)")

## 4. Output characteristic (I&ndash;V$_{ds}$)

Mirrors `plot_Vds_I`, swept at a few gate voltages.

In [ ]:
fig, ax = plt.subplots()
for v_g in [0.0, 0.1, 0.2, 0.3]:
    dev = negforge.Device(v_g=v_g)
    curve = dev.sweep_v_ds(0.0, 0.6, 0.02, self_consistent=False)
    ax.plot(curve.voltage, curve.current, label=f"$V_g$={v_g} V")
ax.set_xlabel(r"$V_{ds}$ (V)")
ax.set_ylabel("current (a.u., see README for units)")
ax.set_title("Output characteristic")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 5. Local density of states (NEGF)

Mirrors `calc_green()`'s `imagesc` plot: the local density of states as a function of position and energy, with the electrostatic potential profile overlaid.

In [ ]:
dev = negforge.Device(v_ds=0.3, v_g=0.2)
dev.solve_self_consistent()
energies, ldos = dev.local_density_of_states()

fig, ax = plt.subplots(figsize=(8, 5))
mesh = ax.pcolormesh(
    dev.positions_nm, energies, ldos, shading="auto", cmap="inferno", vmax=np.percentile(ldos, 99)
)
ax.plot(dev.positions_nm, dev.psi_f, color="cyan", linewidth=1.5, label=r"$\Psi_f$")
ax.set_xlabel("position (nm)")
ax.set_ylabel("energy (eV)")
ax.set_title("Local density of states")
ax.legend(loc="upper right")
fig.colorbar(mesh, ax=ax, label="LDOS (a.u.)")
plt.show()

## 6. Dense vs. recursive NEGF, and parallelism

Every NEGF quantity here (local density of states, injected charge) can be computed two ways:

- `algorithm="recursive"` (default): an O(N) recursive Green's-function sweep. Fast enough to call repeatedly inside the self-consistent loop or a bias sweep.
- `algorithm="dense"`: a literal O(N^3) full-matrix inversion, matching the original MATLAB `inv()` call. Simple and easy to trust, but scales terribly &mdash; kept mainly as a reference to validate the fast path against.

Both give the same answer; only the speed differs. Energy points are also evaluated in parallel across CPU cores for either algorithm, which is why the gap below is smaller than the crate's own benchmark (`crates/negforge-core/examples/benchmark.rs`) shows for a single-threaded run &mdash; see the README's "Performance" section for the full numbers and the reasoning behind both choices.

In [ ]:
import time

# A small device, so the O(N^3) dense path finishes in reasonable time.
small = negforge.Device(v_ds=0.3, v_g=0.2, a=1.0, l_ch=10.0, l_ds=10.0, auto_size_contacts=False)

t0 = time.perf_counter()
small_recursive = small.local_density_of_states(algorithm="recursive")
t_recursive = time.perf_counter() - t0

t0 = time.perf_counter()
small_dense = small.local_density_of_states(algorithm="dense")
t_dense = time.perf_counter() - t0

max_diff = np.max(np.abs(np.array(small_recursive[1]) - np.array(small_dense[1])))
print(f"N = {small.n} sites")
print(f"recursive: {t_recursive * 1000:.2f} ms")
print(f"dense:     {t_dense * 1000:.2f} ms  ({t_dense / t_recursive:.0f}x slower)")
print(f"max |LDOS difference| between algorithms: {max_diff:.2e} (should be ~0)")

## 7. Interactive exploration

Drag the sliders to see how gate and drain bias reshape the (decoupled) potential profile in real time.

In [ ]:
from ipywidgets import interact, FloatSlider


def plot_potential(v_g=0.2, v_ds=0.3):
    dev = negforge.Device(v_g=v_g, v_ds=v_ds)
    dev.calc_potential()
    plt.figure(figsize=(7, 4.5))
    plt.plot(dev.positions_nm, dev.psi_f)
    plt.xlabel("position (nm)")
    plt.ylabel(r"$\Psi_f$ (eV)")
    plt.ylim(-1.0, 1.0)
    plt.grid(alpha=0.3)
    plt.title(f"V$_g$={v_g:.2f} V, V$_{{ds}}$={v_ds:.2f} V")
    plt.show()


interact(
    plot_potential,
    v_g=FloatSlider(min=-0.5, max=1.0, step=0.05, value=0.2),
    v_ds=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.3),
);